# AI Cybersecurity Threat Intelligence System## Full 10-Step Roadmap**Goal:** Any logs -> AI understands -> Detects threats -> Explains risk -> Fast inference API### Architecture`Raw Logs -> Normalization Engine -> MiniLM Semantic Encoder -> Threat Classification -> Severity + Confidence -> Explainability -> Inference API`### Stack| Layer | Technology ||-------|------------|| Data | Pandas || NLP Model | MiniLM (all-MiniLM-L6-v2) || Training | HuggingFace Transformers || Explainability | SHAP || Optimization | ONNX || API | FastAPI || Deployment | HuggingFace Spaces |

## STEP 1 - Understand & Explore Your Dataset### Goal: Discover attack patterns, log distributions, noisy fields, useful features

In [ ]:
# Install dependencies (run once)# !pip install pandas numpy matplotlib seaborn scikit-learn transformers datasets shap torch onnx onnxruntime accelerate -q

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport os, glob, warningswarnings.filterwarnings('ignore')plt.style.use('dark_background')sns.set_theme(style='darkgrid')# Load chunked datasetchunk_dir = 'dataset/chunks'if os.path.exists(chunk_dir):    chunk_files = sorted(glob.glob(os.path.join(chunk_dir, 'data_chunk_*.csv')))    print(f'Loading {len(chunk_files)} chunks...')    dfs = [pd.read_csv(f) for f in chunk_files[:3]]    df = pd.concat(dfs, ignore_index=True)else:    df = pd.read_csv('dataset/cybersecurity_threat_detection_logs.csv')print(f'Dataset shape: {df.shape}')print(f'Columns: {list(df.columns)}')df.head()

In [ ]:
print('=== DATASET OVERVIEW ===')print(f'Rows: {len(df):,}')print(f'Columns: {len(df.columns)}')print(f'Memory: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')print()print('=== NULL VALUES ===')print(df.isnull().sum())print()print('=== THREAT LABEL DISTRIBUTION ===')print(df['threat_label'].value_counts())print()print('=== PROTOCOL DISTRIBUTION ===')print(df['protocol'].value_counts())print()print('=== ACTION DISTRIBUTION ===')print(df['action'].value_counts())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))fig.suptitle('Cybersecurity Dataset Analysis', fontsize=16, color='white')ax = axes[0, 0]threat_counts = df['threat_label'].value_counts()colors = ['#22c55e' if x == 'benign' else '#eab308' if x == 'suspicious' else '#ef4444' for x in threat_counts.index]ax.bar(threat_counts.index, threat_counts.values, color=colors, edgecolor='white')ax.set_title('Threat Label Distribution', color='white')ax.tick_params(colors='white')ax.set_facecolor('#111113')ax = axes[0, 1]proto_counts = df['protocol'].value_counts()ax.barh(proto_counts.index[::-1], proto_counts.values[::-1], color='#3b82f6', edgecolor='white')ax.set_title('Protocol Distribution', color='white')ax.tick_params(colors='white')ax.set_facecolor('#111113')ax = axes[1, 0]df.boxplot(column='bytes_transferred', by='threat_label', ax=ax)ax.set_title('Bytes Transferred by Threat', color='white')ax.set_xlabel('')ax.tick_params(colors='white')ax.set_facecolor('#111113')ax = axes[1, 1]action_threat = pd.crosstab(df['threat_label'], df['action'])action_threat.plot(kind='bar', ax=ax, color=['#22c55e', '#ef4444'])ax.set_title('Action by Threat Label', color='white')ax.tick_params(colors='white')ax.set_facecolor('#111113')plt.tight_layout()plt.savefig('models/dataset_analysis.png', dpi=150, bbox_inches='tight', facecolor='#09090b')plt.show()

In [ ]:
# KEY DISCOVERY: What makes malicious logs different from benign?malicious = df[df['threat_label'] == 'malicious']benign = df[df['threat_label'] == 'benign']print('USER AGENTS - Top Malicious:')print(malicious['user_agent'].value_counts().head(10))print()print('USER AGENTS - Top Benign:')print(benign['user_agent'].value_counts().head(10))print()print('REQUEST PATHS - Top Malicious:')print(malicious['request_path'].value_counts().head(10))print()print('BYTES - Stats by Threat:')print(df.groupby('threat_label')['bytes_transferred'].describe())

## STEP 2 - Build Log Normalization Engine### Goal: Convert ALL logs into canonical semantic format for NLP**Raw:** protocol=TCP, action=blocked, user_agent=Nmap**Normalized:** "Blocked TCP connection detected by firewall log using nmap scanner targeting high-risk path."This is how generalized intelligence emerges across firewall, windows, dns, cloud logs.

In [ ]:
import re, mathclass LogNormalizer:    PROTOCOL_MAP = {        'TCP': 'TCP connection', 'UDP': 'UDP datagram',        'HTTP': 'HTTP request', 'HTTPS': 'HTTPS secure request',        'FTP': 'FTP file transfer', 'SSH': 'SSH session',        'ICMP': 'ICMP ping', 'DNS': 'DNS query'    }    ACTION_MAP = {'allowed': 'permitted', 'blocked': 'blocked', 'denied': 'denied', 'dropped': 'dropped'}    SCANNER_AGENTS = ['nmap', 'sqlmap', 'nikto', 'masscan', 'zap', 'burp', 'dirbuster', 'gobuster', 'wfuzz']        @staticmethod    def detect_scanner(user_agent):        ua_lower = str(user_agent).lower()        for scanner in LogNormalizer.SCANNER_AGENTS:            if scanner in ua_lower:                return scanner        return 'standard browser'        @staticmethod    def classify_path_risk(path):        path_lower = str(path).lower()        if any(x in path_lower for x in ['admin', 'config', 'backup', 'wp-', '.sql', '.env', 'shell']):            return 'high-risk path'        if any(x in path_lower for x in ['login', 'auth', 'secure']):            return 'authentication path'        if any(x in path_lower for x in ['api', 'upload', 'download']):            return 'data operation path'        return 'standard path'        @staticmethod    def bytes_category(b):        b = int(b)        if b < 1000: return 'minimal data transfer'        if b < 10000: return 'small data transfer'        if b < 30000: return 'moderate data transfer'        return 'large data transfer'        @classmethod    def normalize(cls, row):        protocol = cls.PROTOCOL_MAP.get(row.get('protocol', ''), row.get('protocol', 'unknown'))        action = cls.ACTION_MAP.get(row.get('action', ''), row.get('action', 'unknown'))        scanner = cls.detect_scanner(row.get('user_agent', ''))        path_risk = cls.classify_path_risk(row.get('request_path', '/'))        bytes_cat = cls.bytes_category(row.get('bytes_transferred', 0))        log_type = row.get('log_type', 'unknown')                if scanner != 'standard browser':            return f"{action.capitalize()} {protocol} detected by {log_type} log using {scanner} scanner targeting {path_risk} with {bytes_cat}."        return f"{action.capitalize()} {protocol} recorded by {log_type} log accessing {path_risk} with {bytes_cat}."        @classmethod    def normalize_batch(cls, df):        return df.apply(cls.normalize, axis=1)# Testtest_row = df.iloc[0].to_dict()print('RAW:', test_row)print()print('NORMALIZED:', LogNormalizer.normalize(test_row))

In [ ]:
# Apply normalizationprint('Normalizing dataset...')df['normalized_text'] = LogNormalizer.normalize_batch(df)print(f'Normalized {len(df)} logs')for i, text in enumerate(df['normalized_text'].head(5)):    print(f'{i+1}. {text}')

## STEP 3 - Data Cleaning & Label Engineering### Goal: Prepare dataset for transformer training

In [ ]:
print('=== DATA CLEANING ===')print(f'Before: {len(df)} rows')df = df.dropna(subset=['normalized_text', 'threat_label'])print(f'After nulls: {len(df)} rows')df = df.drop_duplicates(subset=['normalized_text', 'source_ip', 'dest_ip'])print(f'After duplicates: {len(df)} rows')LABEL_MAP = {'benign': 0, 'suspicious': 1, 'malicious': 2}df['label'] = df['threat_label'].map(LABEL_MAP)print()print('Label distribution:')print(df['label'].value_counts().sort_index())

## STEP 4 - Add Structured Features### Goal: Combine NLP semantics with structured security metadataTransformers alone are NOT enough. Structured cybersecurity signals are powerful.

In [ ]:
def add_structured_features(df):    df['bytes_log'] = np.log1p(df['bytes_transferred'])    df['bytes_high'] = (df['bytes_transferred'] > 30000).astype(int)        proto_map = {'TCP': 0, 'UDP': 1, 'HTTP': 2, 'HTTPS': 3, 'FTP': 4, 'SSH': 5, 'ICMP': 6, 'DNS': 7}    df['protocol_encoded'] = df['protocol'].map(proto_map).fillna(8).astype(int)        df['action_blocked'] = (df['action'] == 'blocked').astype(int)    df['path_depth'] = df['request_path'].apply(lambda x: len(str(x).split('/')) - 1)        risky = ['admin', 'config', 'backup', 'wp-', '.sql', '.env', 'shell', 'login', 'auth']    df['path_risk_score'] = df['request_path'].apply(lambda x: sum(1 for p in risky if p in str(x).lower()))        scanners = ['nmap', 'sqlmap', 'nikto', 'masscan', 'zap', 'burp']    df['is_scanner'] = df['user_agent'].apply(lambda x: 1 if any(s in str(x).lower() for s in scanners) else 0)        def calc_entropy(s):        s = str(s)        if not s: return 0        prob = [float(s.count(c)) / len(s) for c in set(s)]        return -sum(p * math.log2(p) for p in prob if p > 0)    df['ua_entropy'] = df['user_agent'].apply(calc_entropy)        log_type_map = {'firewall': 0, 'ids': 1, 'application': 2}    df['log_type_encoded'] = df['log_type'].map(log_type_map).fillna(3).astype(int)        df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')    df['hour'] = df['timestamp'].dt.hour.fillna(12).astype(int)    df['is_night'] = ((df['hour'] >= 22) | (df['hour'] <= 5)).astype(int)    return dfdf = add_structured_features(df)feature_cols = ['bytes_log', 'bytes_high', 'protocol_encoded', 'action_blocked',                'path_depth', 'path_risk_score', 'is_scanner', 'ua_entropy',                'log_type_encoded', 'hour', 'is_night']print('Structured features added:', feature_cols)print(df[feature_cols].describe())

## STEP 5 - Tokenization & Embedding Pipeline### Goal: Teach model cybersecurity language using MiniLMMiniLM benefits: Small, Fast inference, Lightweight (Colab-friendly), Semantic understanding, Strong NLP

In [ ]:
from transformers import AutoTokenizerMODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)print(f'Tokenizer: {MODEL_NAME}')print(f'Vocab size: {tokenizer.vocab_size}')print(f'Max length: {tokenizer.model_max_length}')sample = df['normalized_text'].iloc[0]tokens = tokenizer(sample, truncation=True, max_length=128)print(f'Sample: {sample}')print(f'Token count: {len(tokens["input_ids"])}')

In [ ]:
from sklearn.model_selection import train_test_splittrain_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])print(f'Train: {len(train_df)}, Test: {len(test_df)}')print('Train labels:', train_df['label'].value_counts().sort_index().to_dict())train_df.to_csv('models/train_data.csv', index=False)test_df.to_csv('models/test_data.csv', index=False)print('Saved train_data.csv and test_data.csv')

## STEP 6 - Train Threat Classification Model### Goal: Build semantic threat detector with MiniLM

In [ ]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainerfrom datasets import Datasetfrom sklearn.metrics import accuracy_score, precision_recall_fscore_supportimport numpy as npNUM_LABELS = 3LABEL_NAMES = ['benign', 'suspicious', 'malicious']def tokenize_function(examples):    return tokenizer(examples['normalized_text'], truncation=True, max_length=128, padding='max_length')train_dataset = Dataset.from_pandas(train_df[['normalized_text', 'label']])test_dataset = Dataset.from_pandas(test_df[['normalized_text', 'label']])train_dataset = train_dataset.map(tokenize_function, batched=True, batch_size=1000)test_dataset = test_dataset.map(tokenize_function, batched=True, batch_size=1000)train_dataset = train_dataset.remove_columns(['normalized_text']).rename_column('label', 'labels')test_dataset = test_dataset.remove_columns(['normalized_text']).rename_column('label', 'labels')train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])print(f'Train: {len(train_dataset)}, Test: {len(test_dataset)}')

In [ ]:
def compute_metrics(pred):    labels = pred.label_ids    preds = np.argmax(pred.predictions, axis=1)    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')    return {'accuracy': accuracy_score(labels, preds), 'f1': f1, 'precision': precision, 'recall': recall}model = AutoModelForSequenceClassification.from_pretrained(    MODEL_NAME, num_labels=NUM_LABELS, problem_type='single_label_classification')print(f'Parameters: {model.num_parameters():,}')

In [ ]:
training_args = TrainingArguments(    output_dir='models/cyber-threat-model',    num_train_epochs=3,    per_device_train_batch_size=32,    per_device_eval_batch_size=64,    learning_rate=2e-5,    weight_decay=0.01,    evaluation_strategy='epoch',    save_strategy='epoch',    load_best_model_at_end=True,    metric_for_best_model='f1',    fp16=True,    logging_steps=50,    save_total_limit=2,    report_to='none')trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset,                  eval_dataset=test_dataset, compute_metrics=compute_metrics, tokenizer=tokenizer)print('Starting training...')trainer.train()print('Training complete!')

In [ ]:
results = trainer.evaluate()print('=== EVALUATION RESULTS ===')for k, v in results.items():    print(f'{k}: {v:.4f}')trainer.save_model('models/cyber-threat-model')tokenizer.save_pretrained('models/cyber-threat-model')print('Model saved to models/cyber-threat-model/')

## STEP 7 - Add Severity & Confidence Engine### Goal: Convert predictions into SOC-style intelligenceSecurity teams need prioritization, triage, risk ranking - NOT just raw predictions.

In [ ]:
import torchfrom transformers import pipelineclass ThreatIntelligenceEngine:    LABEL_NAMES = ['benign', 'suspicious', 'malicious']    SEVERITY_MAP = {        'benign': {'base': 'Low', 'score': 1},        'suspicious': {'base': 'Medium', 'score': 2},        'malicious': {'base': 'High', 'score': 3}    }        def __init__(self, model_path='models/cyber-threat-model'):        self.classifier = pipeline('text-classification', model=model_path,            tokenizer=model_path, return_all_scores=True,            device=0 if torch.cuda.is_available() else -1)        def _get_severity(self, threat, confidence):        base = self.SEVERITY_MAP.get(threat, {'base': 'Unknown', 'score': 0})        if confidence > 0.95:            severity = 'Critical' if threat == 'malicious' else base['base']        elif confidence > 0.85:            severity = base['base']        elif confidence > 0.70:            severity = 'Low' if base['score'] <= 1 else 'Medium'        else:            severity = 'Informational'        return {'severity': severity, 'base_severity': base['base'], 'score': base['score']}        def _get_explanation(self, text, threat, confidence):        reasons = []        t = text.lower()        if 'scanner' in t:            s = 'nmap' if 'nmap' in t else 'sqlmap' if 'sqlmap' in t else 'security scanner'            reasons.append(f'{s.capitalize()} scanner detected')        if 'blocked' in t or 'denied' in t:            reasons.append('Request blocked/denied by security controls')        if 'high-risk' in t:            reasons.append('Targeting high-risk path (admin/config/backup)')        if 'authentication' in t:            reasons.append('Authentication endpoint targeted')        if 'large data' in t:            reasons.append('Unusually large data transfer')        if not reasons:            reasons.append('Pattern matches known threat signature')        return reasons        def predict(self, log_text):        scores = self.classifier(log_text)[0]        scores_dict = {s['label'].replace('LABEL_', ''): s['score'] for s in scores}        label_scores = {}        for key, val in scores_dict.items():            idx = int(key.split('_')[-1]) if '_' in key else int(key)            label_scores[self.LABEL_NAMES[idx]] = val        threat = max(label_scores, key=label_scores.get)        confidence = label_scores[threat]        severity = self._get_severity(threat, confidence)        explanation = self._get_explanation(log_text, threat, confidence)        return {            'threat': threat, 'confidence': round(confidence, 4),            'confidence_pct': f'{confidence*100:.1f}%',            'severity': severity['severity'],            'all_scores': {k: round(v, 4) for k, v in label_scores.items()},            'explanation': explanation        }# Testengine = ThreatIntelligenceEngine()test_logs = [    'Blocked TCP connection detected by firewall log using nmap scanner targeting high-risk path.',    'Permitted HTTP request recorded by application log accessing standard path.',    'Blocked HTTPS secure request detected by ids log using sqlmap scanner targeting authentication path.']for log in test_logs:    r = engine.predict(log)    print(f"Threat: {r['threat']} | Confidence: {r['confidence_pct']} | Severity: {r['severity']}")    print(f"  Explanation: {r['explanation']}")    print(f"  Scores: {r['all_scores']}")    print('-' * 60)

## STEP 8 - Add Explainability Layer (SHAP)### Goal: Make AI explain WHY it flagged a threatExplainable AI is CRITICAL in cybersecurity.

In [ ]:
import shapfrom sklearn.ensemble import RandomForestClassifierfeature_cols = ['bytes_log', 'bytes_high', 'protocol_encoded', 'action_blocked',                'path_depth', 'path_risk_score', 'is_scanner', 'ua_entropy',                'log_type_encoded', 'hour', 'is_night']X_train, y_train = train_df[feature_cols].values, train_df['label'].valuesX_test, y_test = test_df[feature_cols].values, test_df['label'].valuesrf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)rf.fit(X_train, y_train)print(f'RF accuracy: {rf.score(X_test, y_test):.4f}')explainer = shap.TreeExplainer(rf)shap_values = explainer.shap_values(X_test[:1000])shap.summary_plot(shap_values, X_test[:1000], feature_names=feature_cols, show=False, plot_type='bar')plt.savefig('models/shap_importance.png', dpi=150, bbox_inches='tight', facecolor='#09090b')plt.close()import joblibjoblib.dump(rf, 'models/rf_explainer.pkl')joblib.dump(feature_cols, 'models/feature_cols.pkl')print('Saved RF explainer')

## STEP 9 - Optimize For Fast Inference### Goal: Export to ONNX for production deployment

In [ ]:
from transformers import AutoModelForSequenceClassificationimport torchmodel = AutoModelForSequenceClassification.from_pretrained('models/cyber-threat-model')model.eval()dummy = tokenizer('Blocked TCP connection detected by firewall log using nmap scanner.',                  return_tensors='pt', truncation=True, max_length=128, padding='max_length')torch.onnx.export(model, (dummy['input_ids'], dummy['attention_mask']),    'models/cyber-threat-model.onnx',    input_names=['input_ids', 'attention_mask'], output_names=['logits'],    dynamic_axes={'input_ids': {0: 'batch_size'}, 'attention_mask': {0: 'batch_size'}, 'logits': {0: 'batch_size'}},    opset_version=14)import osprint(f'ONNX size: {os.path.getsize("models/cyber-threat-model.onnx")/1e6:.1f} MB')

In [ ]:
import time, onnxruntime as ortsession = ort.InferenceSession('models/cyber-threat-model.onnx')test_texts = [    'Blocked TCP connection detected by firewall log using nmap scanner targeting high-risk path.',    'Permitted HTTP request recorded by application log accessing standard path.',    'Blocked HTTPS secure request detected by ids log using sqlmap scanner.'] * 10start = time.time()for text in test_texts:    tokens = tokenizer(text, return_tensors='np', truncation=True, max_length=128, padding='max_length')    session.run(None, {'input_ids': tokens['input_ids'], 'attention_mask': tokens['attention_mask']})elapsed = time.time() - startprint(f'{len(test_texts)} inferences in {elapsed:.2f}s')print(f'Average: {elapsed/len(test_texts)*1000:.1f}ms per inference')print(f'Throughput: {len(test_texts)/elapsed:.1f} inferences/sec')

## STEP 10 - Build Production Inference System### Goal: Deploy-ready inference + HuggingFace upload

In [ ]:
def predict_security_log(raw_log, normalizer=None, engine=None):    if normalizer is None: normalizer = LogNormalizer    normalized = normalizer.normalize(raw_log)    if engine is None: engine = ThreatIntelligenceEngine()    result = engine.predict(normalized)    result['raw_log'] = raw_log    return resulttest_log = {    'protocol': 'TCP', 'action': 'blocked', 'user_agent': 'Nmap Scripting Engine',    'request_path': '/admin/config', 'bytes_transferred': 45000, 'log_type': 'firewall'}result = predict_security_log(test_log, LogNormalizer, engine)print('=== PRODUCTION OUTPUT ===')print(f"Threat: {result['threat']}")print(f"Confidence: {result['confidence_pct']}")print(f"Severity: {result['severity']}")print(f"Explanation: {result['explanation']}")

In [ ]:
# HuggingFace Hub uploadfrom huggingface_hub import HfApiimport jsonmodel_card = '''---language: enlicense: mittags: [cybersecurity, threat-detection, text-classification, security, nlp, minilm]pipeline_tag: text-classification---# Cybersecurity Threat Intelligence ModelSemantic threat detection model built on MiniLM (all-MiniLM-L6-v2).## Usage`pythonfrom transformers import pipelineclassifier = pipeline('text-classification', model='your-username/cyber-threat-model')result = classifier('Blocked TCP connection detected using nmap scanner.')`'''with open('models/README.md', 'w') as f:    f.write(model_card)config = {'model_name': MODEL_NAME, 'num_labels': NUM_LABELS, 'label_names': LABEL_NAMES,          'label_map': LABEL_MAP, 'max_length': 128, 'version': '1.0.0'}with open('models/config.json', 'w') as f:    json.dump(config, f, indent=2)print('Model card and config saved')print()print('To upload to HuggingFace:')print('1. pip install huggingface_hub')print('2. huggingface-cli login')print('3. api = HfApi()')print('4. api.create_repo(repo_id="your-username/cyber-threat-model", repo_type="model")')print('5. api.upload_folder(folder_path="models/cyber-threat-model", repo_id="your-username/cyber-threat-model")')

## Summary: Final System Architecture`Raw Logs -> LogNormalizer -> Canonical semantic text    -> MiniLM Encoder -> Semantic embeddings    -> Threat Classification -> benign/suspicious/malicious    -> Severity Engine -> Critical/High/Medium/Low    -> Explainability -> SHAP + rule-based reasons    -> Inference API -> FastAPI endpoint`### Files Produced- models/cyber-threat-model/ - Trained MiniLM model- models/cyber-threat-model.onnx - Optimized ONNX model- models/rf_explainer.pkl - SHAP explainer- models/feature_cols.pkl - Feature column names- models/config.json - Model configuration- models/README.md - HuggingFace model card- models/train_data.csv / 	est_data.csv - Split datasets